# Complete and Intuitive Guide to the Kalman Filter (KF)

### A Step-by-Step "From Scratch" Explanation for Mobile Robot Localization (Duckiebot)

Welcome to this notebook! If you have opened this document, you have probably already seen formulas full of matrices, transposes, Gaussians, and Greek letters that might look intimidating.

**Don't worry:** in this guide, we will break down **every single concept, equation, and line of code** from the original Kalman Filter notebook piece by piece, starting from scratch. We will use a **single concrete numerical example** based on our **Duckiebot** moving across the floor of a room.

By the end of this notebook, you will not only understand every single mathematical formula, but you will also be able to implement and visualize it confidently in Python.

---

### The Big Question: Why Do We Need the Kalman Filter?

Imagine driving your Duckiebot across the floor. To know where the robot is at every second, you have two tools at your disposal, **both of which are imperfect**:

1. **The Physics of Motion (Odometry / Dead Reckoning):**
* You know the robot's kinematics and how much current you send to the wheel motors (e.g., "make 5 rotations per second"). If you multiply the rotations by the wheel's circumference, you can estimate how many centimeters the robot has traveled over time starting from its initial state.
* **The problem:** In the real world, we always have to deal with imperfect models. Wheels **slip** on parquet or on the track, the floor is not perfectly smooth, and mechanical parts have tolerances and friction.
* **Consequence:** It is like walking with your eyes closed while counting steps: at first you know where you are, but after 20 steps you will bump into a wall because small slippage errors **accumulate indefinitely** (a phenomenon known as *drift*).


2. **External Sensors (Camera, ToF/Laser Sensors, GPS):**
* Robots are equipped with various sensors that can help us estimate their position (for example, by measuring the distance from walls or reading an AprilTag).
* **The problem:** Sensors are also imperfect and affected by **noise** (light reflections, electronic jitter, finite resolution).
* **Consequence:** Measurements constantly "bounce" around the true value. One instant the sensor says you are at $50\text{ cm}$, the next instant at $53\text{ cm}$, then at $48\text{ cm}$, even if the robot is stationary!



### The Superpower of the Kalman Filter: *Sensor Fusion*

Can we combine sensor data with the physical models we already have to better estimate the robot's state? **This is exactly where the Kalman Filter (KF) helps us!**
Given new data about the robot's state along with a measure of how informative it is (its *covariance*), the KF fuses this information by calculating moment by moment **whom to trust more**, producing a final estimate that is **much more accurate and stable than either source taken alone**.

---

## The Three Fundamental Assumptions of the Kalman Filter: Formula-by-Formula Explanation with Numerical Examples

In the classic Kalman Filter, we make **three fundamental mathematical assumptions** that govern the entire algorithm. Let's break them down piece by piece, understanding the physical meaning of every single symbol and accompanying each formula with a **concrete numerical example** on our **Duckiebot**.

---

### 1. Linear Process with Additive Gaussian Noise:

$$\mathbf{x}_{t} = A_t \mathbf{x}_{t-1} + B_t \mathbf{u}_{t} + \mathbf{w}_{t} \quad \text{where} \quad \mathbf{w}_{t} \sim \mathcal{N}(0, Q_t)$$

#### What is this formula for?

It answers the physical question: *"How does the robot's true position in the world evolve from one second to the next when we send commands to the motors?"*

#### Breaking down every single piece:

* **$\mathbf{x}_t$ and $\mathbf{x}_{t-1}$ (True State Vector):** Represents the robot's pose at the current time step $t$ and at the previous time step $t-1$ (e.g., floor coordinates $[x, y]^T$).
* **$A_t$ (State Transition Matrix):** Describes what the robot does **if we do not touch the controls** (inertia or natural motion). If the Duckiebot, left without commands, stays exactly where it was, $A_t$ is simply the identity matrix $I$ (multiplying by 1):

$$A_t = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$$


* **$\mathbf{u}_t$ (Control Vector):** The command we send to the motors (for instance, the commanded wheel velocity: $v_x = 5\text{ cm/s}, v_y = 0$).
* **$B_t$ (Control Matrix):** The "mechanical translator" that converts commands $\mathbf{u}_t$ into actual displacement on the track (e.g., multiplies velocity by the time interval $\Delta t = 1\text{ s}$ to get centimeters traveled):

$$B_t = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$$


* **$\mathbf{w}_t$ (Process Noise):** The unpredictability of the real world! Wheels can slip on the floor, the floor has tiny bumps, and motors have slight mechanical tolerances.
* **$\mathbf{w}_t \sim \mathcal{N}(0, Q_t)$:** Means this noise follows a normal (Gaussian) distribution with:
* **Mean $0$:** We don't know whether the robot will slip to the right or to the left, but on average, random errors cancel out (the robot does not systematically pull to one side).
* **Covariance $Q_t$:** Represents how "slippery" the floor is. The larger the values in $Q_t$, the less we can trust the motion estimate.


* **Why is it called "Linear with Additive Noise"?**
* *Linear:* There are no complicated curved functions (like $\sin, \cos$, or squares): everything is simple matrix and vector multiplication.
* *Additive:* The noise $\mathbf{w}_t$ enters the equation as a simple addition ($+$ at the end).



#### Concrete Numerical Example (Formula 1):

Imagine the Duckiebot starts stationary at the origin $\mathbf{x}_0 = \begin{bmatrix} 0 \\ 0 \end{bmatrix}\text{ cm}$.
We send a command to move forward along $x$ at $5\text{ cm/s}$ for $\Delta t = 1\text{ s}$, so $\mathbf{u}_1 = \begin{bmatrix} 5 \\ 0 \end{bmatrix}$.
The covariance of floor slippage is known: $Q = \begin{bmatrix} 0.3 & 0 \\ 0 & 0.3 \end{bmatrix}$.

During this second, the wheels slip slightly and an actual disturbance $\mathbf{w}_1 = \begin{bmatrix} +0.4 \\ -0.1 \end{bmatrix}\text{ cm}$ occurs.
The true position where the robot physically ends up is:


$$\mathbf{x}_1 = A_1 \mathbf{x}_0 + B_1 \mathbf{u}_1 + \mathbf{w}_1 = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} \begin{bmatrix} 0 \\ 0 \end{bmatrix} + \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} \begin{bmatrix} 5 \\ 0 \end{bmatrix} + \begin{bmatrix} +0.4 \\ -0.1 \end{bmatrix} = \begin{bmatrix} 5.4 \\ -0.1 \end{bmatrix}\text{ cm}$$


*Instead of being at exactly $(5, 0)$, due to the real noise $\mathbf{w}_1$, the robot physically ended up at $(5.4, -0.1)$!*

---

### 2. Linear Measurement Model with Additive Gaussian Noise:

$$\mathbf{z}_t = H_t \mathbf{x}_t + \mathbf{n}_t \quad \text{where} \quad \mathbf{n}_t \sim \mathcal{N}(0, R_t)$$

#### What is this formula for?

It answers the question: *"What do external sensors (camera, laser) physically read given the robot's true position $\mathbf{x}_t$?"*

#### Breaking down every single piece:

* **$\mathbf{z}_t$ (Measurement Vector):** The raw reading provided by the sensors at time $t$ (for example, a laser beam aimed at the wall returning $8\text{ cm}$, or coordinates detected by an AprilTag).
* **$\mathbf{x}_t$ (True State):** Where the robot is actually located in space.
* **$H_t$ (Observation Matrix or "Translator"):** Projects the true state into sensor space. It adapts variables and units of measurement:
* *Example:* If the state has 2 coordinates $[x, y]^T$, but the laser sensor only measures frontal distance along $x$, the matrix $H = \begin{bmatrix} 1 & 0 \end{bmatrix}$ acts as a filter and extracts only $x$, allowing us to compare homogeneous quantities:

$$H \mathbf{x}_t = \begin{bmatrix} 1 & 0 \end{bmatrix} \begin{bmatrix} x \\ y \end{bmatrix} = x$$




* **$\mathbf{n}_t$ (Measurement Noise):** The sensor's intrinsic error (optical blur, light reflections, electrical jitter in circuits).
* **$\mathbf{n}_t \sim \mathcal{N}(0, R_t)$:** Indicates that the sensor error is Gaussian:
* **Mean $0$:** The sensor is properly calibrated (it does not add a fixed systematic bias of $+5\text{ cm}$).
* **Covariance $R_t$:** The sensor uncertainty specified by the manufacturer. The smaller $R_t$ is, the more accurate and reliable the sensor is.


* **Why is it called "Linear with Additive Noise"?** Because the expected measurement is a linear combination of the state ($H_t \mathbf{x}_t$) to which noise is simply added ($+ \mathbf{n}_t$).

#### Concrete Numerical Example (Formula 2):

The Duckiebot is actually at the true position just calculated: $\mathbf{x}_1 = \begin{bmatrix} 5.4 \\ -0.1 \end{bmatrix}\text{ cm}$.
The robot is equipped with two wall-distance sensors (one along $x$ and one along $y$), so the measurement matrix is the identity matrix:


$$H = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$$


The sensor noise covariance provided by the manufacturer is $R = \begin{bmatrix} 0.75 & 0 \\ 0 & 0.60 \end{bmatrix}$.

When the sensors fire their beams, optical and electrical interference generates an instantaneous noise $\mathbf{n}_1 = \begin{bmatrix} +0.6 \\ -0.3 \end{bmatrix}\text{ cm}$.
The raw measurement that the onboard computer physically receives is:


$$\mathbf{z}_1 = H \mathbf{x}_1 + \mathbf{n}_1 = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} \begin{bmatrix} 5.4 \\ -0.1 \end{bmatrix} + \begin{bmatrix} +0.6 \\ -0.3 \end{bmatrix} = \begin{bmatrix} 5.4 + 0.6 \\ -0.1 - 0.3 \end{bmatrix} = \begin{bmatrix} 6.0 \\ -0.4 \end{bmatrix}\text{ cm}$$


*The sensor returns $(6.0, -0.4)$, a value close to reality $(5.4, -0.1)$ but perturbed by reading noise $\mathbf{n}_1$!*

---

### 3. Gaussian Prior Distribution (*Prior Belief*):

$$bel(\mathbf{x}_0) = \mathcal{N}(\mu_0, \Sigma_0)$$

#### What is this formula for?

It answers the question: *"Even before turning on the motors at time $t=0$, what do we know about where the robot is and how certain are we?"*

#### Breaking down every single piece:

* **$bel(\mathbf{x}_0)$ (*Belief*):** Our probabilistic belief about the initial state.
* **$\mathcal{N}$ (Normal or Gaussian Distribution):** States that uncertainty has the classic bell curve shape. The value at the center of the bell is the most probable, while probabilities fade away as we move further from the center.
* **$\mu_0$ (Initial Mean Vector):** The exact point in space where we believe the robot is located (e.g., $[0, 0]^T$ if we place it on the track origin).
* **$\Sigma_0$ (Initial Covariance Matrix):** Measures the spread of our initial doubt:
* *If you place the Duckiebot using a millimeter ruler:* You are very confident, so you set very small variances (e.g., $\Sigma_0 = \text{diag}(0.1, 0.1)$).
* *If you place the robot randomly in the room without looking:* You set huge numbers (e.g., $\Sigma_0 = \text{diag}(1000, 1000)$), telling the filter: *"Don't trust my initial estimate $\mu_0$, trust the very first sensor reading immediately!"*



#### Concrete Numerical Example (Formula 3):

* **Case A (Starting with precise manual placement):**
You place the Duckiebot exactly on the starting square $(0, 0)$ using a ruler or template on the track.
Your initial mean estimate and uncertainty matrix are:

$$\mu_0 = \begin{bmatrix} 0 \\ 0 \end{bmatrix}\text{ cm}, \quad \Sigma_0 = \begin{bmatrix} 0.1 & 0 \\ 0 & 0.1 \end{bmatrix}\text{ cm}^2$$



*Practical meaning:* You are roughly $99\%$ certain that the robot is between $-0.3\text{ cm}$ and $+0.3\text{ cm}$ around zero (the $3\sigma = 3 \cdot \sqrt{0.1} \approx 0.95$ rule).
* **Case B (Starting blind without knowing where the robot is):**
If you turn on the robot placed randomly in a $10 \times 10\text{ meter}$ room, you set:

$$\mu_0 = \begin{bmatrix} 5 \\ 5 \end{bmatrix}\text{ m}, \quad \Sigma_0 = \begin{bmatrix} 1000 & 0 \\ 0 & 1000 \end{bmatrix}\text{ m}^2$$



*Practical meaning:* You mathematically tell the Kalman Filter: *"My initial estimate $\mu_0$ is worthless—give $100\%$ of the trust to the first reading from the sensor!"*

---

### The "Mathematical Secret" of These 3 Assumptions

> **The Fundamental Theorem of Gaussians:**
> If you start from a Gaussian bell-shaped uncertainty (**Assumption 3**) and apply only linear transformations with additive Gaussian noise (**Assumptions 1 and 2**), **the result ALWAYS remains a perfect Gaussian distribution!**
> This means the robot does not need to store monstrous probability shapes or compute complex integrals: to know the entire state of the world, it only needs two very simple objects:
> 1. **The Mean $\mu$:** the center of the bell (the estimated position);
> 2. **The Covariance $\Sigma$:** the width and tilt of the uncertainty ellipse.
> 
> 
> The entire algorithm reduces to fast, real-time algebraic matrix operations!

---

# Step 1: What Do We Want to Know About the Robot? The State Vector ($\mathbf{x}_t$)

When we were in the one-dimensional (1D) case—for example, a toy car on a straight track—a single number was enough (e.g., $x = 50\text{ cm}$).

However, on a real floor, our Duckiebot moves in a two-dimensional plane $(x, y)$.

Therefore, instead of a single number, we group the coordinates into a "vertical list" called the **state vector $\mathbf{x}_t$**:

$$\mathbf{x}_t = \begin{bmatrix} x \\ y \end{bmatrix}$$

### Concrete Numerical Example:

If we think that at the current instant the Duckiebot is at $10\text{ cm}$ along the $x$-axis and $20\text{ cm}$ along the $y$-axis, we write the mean vector $\mu_t$ as:

$$\mu_t = \begin{bmatrix} 10 \\ 20 \end{bmatrix}$$

> **Consequence:**
> All the operations we previously performed with single numbers must now be done with tables of numbers (vectors and matrices) operating on $x$ and $y$ simultaneously.

---

# Step 2: How Do We Measure Our Doubt in 2D? The Covariance Matrix ($\Sigma$)

In the 1D case, we had a single variance $\sigma^2$. Now that we have two variables ($x$ and $y$), our doubt becomes a $2 \times 2$ square table called the **Covariance Matrix $\Sigma$** (often also denoted by $P$ in robotics literature):

$$\Sigma = \begin{bmatrix} \sigma_x^2 & \text{cov}(x,y) \\ \text{cov}(y,x) & \sigma_y^2 \end{bmatrix}$$

---

### The Semantic Meaning of Every Single Entry:

1. **The Main Diagonal ($\sigma_x^2$ and $\sigma_y^2$) — The Variances:**
* These are the standard variances along $x$ and $y$ taken individually.
* They tell you how wide and how tall your "circle of doubt" (the **uncertainty ellipse**) is around the robot.
* If $\sigma_x^2$ is very large, we know almost nothing about where $x$ is; if it is close to zero, we know $x$ with millimeter precision.


2. **The Off-Diagonal Elements ($\text{cov}(x,y)$ and $\text{cov}(y,x)$) — The Covariances:**
* The matrix is always symmetric, so $\text{cov}(x,y) = \text{cov}(y,x)$.
* They measure the **coupling between errors**: *"If I overestimated $x$ by a few centimeters, is it likely that I also overestimated $y$?"*
* **Covariance $= 0$:** Independent errors. Being wrong about $x$ gives no clue about $y$.
* **Covariance $> 0$ (positive):** When $x$ increases or has a positive error, $y$ tends to increase as well.
* **Covariance $< 0$ (negative):** When $x$ increases, $y$ tends to decrease.



---

### Numerical Examples and Geometric Intuition:

#### Numerical Example 1 (Independent errors, $\text{cov} = 0$):

$$\Sigma = \begin{bmatrix} 4 & 0 \\ 0 & 9 \end{bmatrix}$$

* The doubt on $x$ is $4$ (standard deviation $\sigma_x = 2\text{ cm}$).
* The doubt on $y$ is $9$ (standard deviation $\sigma_y = 3\text{ cm}$).
* Geometrically, this is an **upright ellipse**, aligned perfectly with the Cartesian axes.

#### Numerical Example 2 (The robot moves diagonally at $45^\circ$, $\text{cov} > 0$):

Imagine the Duckiebot is moving diagonally at $45^\circ$ (where $x$ and $y$ increase together) and the wheels slip slightly forward:

* If the robot slipped further ahead than expected, **both $x$ and $y$ will simultaneously be larger than expected!**
* Then the matrix becomes, for example:

$$\Sigma = \begin{bmatrix} 4 & 3 \\ 3 & 4 \end{bmatrix}$$


* That positive $+3$ **tilts the ellipse at $45^\circ$**!

> **Remarkable Consequence:**
> If there is a non-zero number off the diagonal, **measuring only the $x$ coordinate with a sensor is enough to automatically discover how much $y$ was off too!**
> In the Kalman Filter, we use three covariance matrices:
> * $\Sigma_0$: for the initial state uncertainty;
> * $Q_t$: for the uncertainty introduced by wheel slippage during motion;
> * $R_t$: for sensor uncertainty and noise.
> 
> 

---

# Step 3: Where Do We Start? The Initial State — $bel(\mathbf{x}_0)$

Before turning on the motors at the initial time $t=0$, we must tell the filter where the robot is and how certain we are about it:

$$bel(\mathbf{x}_0) = \mathcal{N}(\mu_0, \Sigma_0)$$

* **Semantic Meaning:** $bel(\mathbf{x}_0)$ (*belief*) states that initially, our probabilistic knowledge of the position is described by a Gaussian bell centered at $\mu_0$ with uncertainty matrix $\Sigma_0$.
* **$\mu_0$ (Initial Mean):** Our best initial estimate (e.g., $[0, 0]^T$ if starting from the track origin).
* **$\Sigma_0$ (Initial Covariance):** How much we trust this estimate.

### Numerical Example:

You place the Duckiebot manually right on the origin $(0,0)$ using a ruler or template (you are very confident, small variance $= 0.1$):

$$\mu_0 = \begin{bmatrix} 0 \\ 0 \end{bmatrix}, \quad \Sigma_0 = \begin{bmatrix} 0.1 & 0 \\ 0 & 0.1 \end{bmatrix}$$

> **Consequence:**
> If instead you powered on the robot without knowing at all where it was on the track, you would set huge numbers on the diagonal of $\Sigma_0$ (e.g., $1000$), telling the filter:
> *"Don't trust my initial estimate $\mu_0$, trust the very first camera frame or sensor reading immediately!"*

---

# Step 4: The Robot Moves — The Process Model and Prediction (*Predict Step*)

Now we power the motors for 1 time interval $\Delta t = 1\text{ s}$. How does the state change in the real world from $t-1$ to $t$? The first fundamental equation tells us:

$$\mathbf{x}_{t} = A_t \mathbf{x}_{t-1} + B_t \mathbf{u}_{t} + \mathbf{w}_{t} \quad \text{where} \quad \mathbf{w}_{t} \sim \mathcal{N}(0, Q_t)$$

Let's break down every letter with a concrete example on the Duckiebot:

* **$A_t$ (State Transition Matrix):** Tells what the state does if we don't touch the controls. If the robot stays right where it was when left idle, $A_t$ is the identity matrix $I$:

$$A_t = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$$


* **$\mathbf{u}_t$ (Control Vector):** The command you send to the motors (for example, commanded velocities $v_x = 1\text{ m/s}, v_y = 1\text{ m/s}$):

$$\mathbf{u}_t = \begin{bmatrix} 1 \\ 1 \end{bmatrix}$$


* **$B_t$ (Control Matrix):** Translates the command $\mathbf{u}_t$ into real displacement (e.g., multiplies velocity by time $\Delta t = 1\text{ s}$):

$$B_t = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$$


* **$\mathbf{w}_t \sim \mathcal{N}(0, Q_t)$ (Process Noise and Covariance $Q_t$):** The wheels slip! We don't know by how much they will slip this time (noise has mean $0$, meaning it doesn't always pull right or always left), but we know the slippage uncertainty matrix $Q_t$:

$$Q_t = \begin{bmatrix} 0.3 & 0 \\ 0 & 0.3 \end{bmatrix}$$



---

### The Two Equations of the *Prediction Step*

In the prediction phase, we obtain a provisional estimate called the **prior belief** $\bar{bel}(\mathbf{x}_t) = \mathcal{N}(\bar{\mu}_t, \bar{\Sigma}_t)$.
The **bar over the symbols** ($\bar{\mu}_t, \bar{\Sigma}_t$) indicates that this estimate is computed **blindfolded**, before looking at external sensors:

$$\mathbf{1.} \quad \bar{\mu}_{t} = A_t \mu_{t-1} + B_t \mathbf{u}_{t}$$

$$\mathbf{2.} \quad \bar{\Sigma}_{t} = A_t \Sigma_{t-1} A_t^T + Q_t$$

* **Why is $\mathbf{w}_t$ missing from the mean (1)?** Because we assume zero-mean noise: on average, random slips to the left and right cancel each other out.
* **Why does the uncertainty (2) have $A \Sigma A^T + Q_t$?** Multiplying on the left by $A$ and on the right by its transpose $A^T$ is the algebraic rule for transforming a covariance matrix. Then we **add $+ Q_t$**: moving in the dark inevitably increases doubt!

---

### Step-by-Step Numerical Calculation (from time $t=0$ to $t=1$):

Starting from $\mu_0 = \begin{bmatrix} 0 \\ 0 \end{bmatrix}$ and $\Sigma_0 = \begin{bmatrix} 0.1 & 0 \\ 0 & 0.1 \end{bmatrix}$ with $\mathbf{u}_1 = \begin{bmatrix} 1 \\ 1 \end{bmatrix}$:

1. **New predicted position:**

$$\bar{\mu}_1 = A_t \mu_0 + B_t \mathbf{u}_1 = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} \begin{bmatrix} 0 \\ 0 \end{bmatrix} + \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} \begin{bmatrix} 1 \\ 1 \end{bmatrix} = \begin{bmatrix} 1 \\ 1 \end{bmatrix}$$


2. **New predicted uncertainty ($Q_t$ is added!):**

$$\bar{\Sigma}_1 = A_t \Sigma_0 A_t^T + Q_t = \begin{bmatrix} 0.1 & 0 \\ 0 & 0.1 \end{bmatrix} + \begin{bmatrix} 0.3 & 0 \\ 0 & 0.3 \end{bmatrix} = \begin{bmatrix} 0.4 & 0 \\ 0 & 0.4 \end{bmatrix}$$



> **What would happen if we only continued with the Prediction step?**
> 1. The mean $\bar{\mu}_t$ would follow odometry, accumulating real error (drift).
> 2. The matrix $\bar{\Sigma}_t$ would keep adding $+Q_t$ at every step, inflating the ellipse of doubt until it covered the entire room!
> 
> 

---

# Step 5: The Sensor Looks Outside — The Measurement Model (*Observation Model*)

Having reached $(1, 1)$, the Duckiebot "opens its eyes" and turns on an external sensor to check where it is. Here is the second fundamental equation:

$$\mathbf{z}_t = H_t \mathbf{x}_t + \mathbf{n}_t \quad \text{where} \quad \mathbf{n}_t \sim \mathcal{N}(0, R_t)$$

### What is $H_t$ (Observation Matrix) Really For?

$H_t$ acts as a **"translator/selector"** that projects the robot's state into sensor space.

* **Example A (1D laser sensor measuring only the $x$ coordinate):**
The state $\mathbf{x}_t$ has two numbers ($x$ and $y$), but the laser sensor reads only 1 number ($x$).
How do you compare them? The matrix $H_t = \begin{bmatrix} 1 & 0 \end{bmatrix}$ extracts only $x$:

$$H_t \bar{\mu} = \begin{bmatrix} 1 & 0 \end{bmatrix} \begin{bmatrix} x \\ y \end{bmatrix} = 1 \cdot x + 0 \cdot y = x$$


* **Example B (2D sensors in our notebook):**
In our exercise, the Duckiebot has sensors measuring the distance from both the $x$-wall and the $y$-wall:

$$H_t = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$$


* **$\mathbf{z}_t$ (Actual Measurement):** The raw reading provided by the sensor (e.g., $\mathbf{z}_1 = \begin{bmatrix} 1.3 \\ 0.8 \end{bmatrix}$).
* **$\mathbf{n}_t \sim \mathcal{N}(0, R_t)$ (Measurement Noise and Covariance $R_t$):** The sensor also has an optical/electrical margin of error, described by $R_t$:

$$R_t = \begin{bmatrix} 0.75 & 0 \\ 0 & 0.60 \end{bmatrix}$$


* The smaller $R_t$ is, the more accurate the sensor is; the larger it is, the noisier the measurements are.



---

# Step 6: The Optimal Update — The *Update Step*

In the **Update step**, the Duckiebot compares the position it predicted blindfolded ($\bar{\mu}_t, \bar{\Sigma}_t$) with what the sensor reads ($\mathbf{z}_t, R_t$).
The goal is to obtain the definitive, most accurate possible estimate: $bel(\mathbf{x}_t) = \mathcal{N}(\mu_t, \Sigma_t)$ (no longer having the bar over the symbols).

The 4 steps of the Update:

1. **Measurement Acquisition:** We read $\mathbf{z}_t$.
2. **Prediction Residual (or *Innovation*):**

$$\delta_{\mu} = \mathbf{z}_{t} - H_t \bar{\mu}_{t}$$


$$\delta_{\Sigma} = H_t \bar{\Sigma}_{t} H_t^T + R_t$$


* $\delta_\mu$ is the **surprise**: how much the actual measurement differs from what we expected. If $\delta_\mu = 0$, the measurement completely confirms our prediction.
* $\delta_\Sigma$ is the **total uncertainty** in this comparison (it sums the motion error projected into sensor space and the sensor's own noise).


3. **Kalman Gain ($K_t$):**

$$K_{t} = \bar{\Sigma}_{t} H_t^T \delta_{\Sigma}^{-1} = \frac{\bar{\Sigma}_{t} H_t^T}{H_t \bar{\Sigma}_{t} H_t^T + R_t}$$


* $K_t$ is the **trust scale**: it decides how much to shift our estimate toward the sensor relative to the prediction.
* **If the sensor is ultra-precise ($R_t \to 0$):** $K_t \to H_t^{-1}$. The filter blindly trusts the sensor.
* **If the sensor is extremely noisy ($R_t \to \infty$):** $K_t \to 0$. The filter ignores the sensor and keeps the wheel prediction.


4. **Belief Update ($\mu_t$ and $\Sigma_t$):**

$$\mu_{t} = \bar{\mu}_{t} + K_t \delta_{\mu}$$


$$\Sigma_{t} = \bar{\Sigma}_{t} - K_t H_t \bar{\Sigma}_{t}$$


* The new mean $\mu_t$ is corrected toward the measurement proportionally to $K_t$.
* The new uncertainty $\Sigma_t$ **decreases** (notice the minus sign!): having acquired new information from the outside world, our doubt always drops.



### Concrete Numerical Calculation of the Update (Continuing from Step 4):

* Prediction: $\bar{\mu}_1 = \begin{bmatrix} 1 \\ 1 \end{bmatrix}$, $\quad \bar{\Sigma}_1 = \begin{bmatrix} 0.4 & 0 \\ 0 & 0.4 \end{bmatrix}$
* Sensor matrix: $H = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$
* Sensor noise: $R = \begin{bmatrix} 0.75 & 0 \\ 0 & 0.60 \end{bmatrix}$
* Sensor measurement: $\mathbf{z}_1 = \begin{bmatrix} 1.3 \\ 0.8 \end{bmatrix}$

Let's compute:

1. **Residual:**

$$\delta_\mu = \begin{bmatrix} 1.3 \\ 0.8 \end{bmatrix} - \begin{bmatrix} 1 \\ 1 \end{bmatrix} = \begin{bmatrix} +0.3 \\ -0.2 \end{bmatrix}$$


2. **Residual Covariance:**

$$\delta_\Sigma = \begin{bmatrix} 0.4 & 0 \\ 0 & 0.4 \end{bmatrix} + \begin{bmatrix} 0.75 & 0 \\ 0 & 0.60 \end{bmatrix} = \begin{bmatrix} 1.15 & 0 \\ 0 & 1.00 \end{bmatrix}$$


3. **Kalman Gain $K$:**

$$K = \begin{bmatrix} 0.4 & 0 \\ 0 & 0.4 \end{bmatrix} \begin{bmatrix} \frac{1}{1.15} & 0 \\ 0 & \frac{1}{1.00} \end{bmatrix} \approx \begin{bmatrix} 0.348 & 0 \\ 0 & 0.400 \end{bmatrix}$$


4. **New Corrected Estimated Position $\mu_1$:**

$$\mu_1 = \begin{bmatrix} 1 \\ 1 \end{bmatrix} + \begin{bmatrix} 0.348 \cdot (+0.3) \\ 0.400 \cdot (-0.2) \end{bmatrix} = \begin{bmatrix} 1.104 \\ 0.920 \end{bmatrix}$$



*(The estimate settled smoothly between the prediction $(1, 1)$ and the measurement $(1.3, 0.8)$!)*
5. **New Uncertainty $\Sigma_1$:**

$$\Sigma_1 = \begin{bmatrix} 0.4 & 0 \\ 0 & 0.4 \end{bmatrix} - \begin{bmatrix} 0.348 \cdot 0.4 & 0 \\ 0 & 0.400 \cdot 0.4 \end{bmatrix} = \begin{bmatrix} 0.261 & 0 \\ 0 & 0.240 \end{bmatrix}$$



*(Uncertainty dropped from $0.4$ down to roughly $0.25$!)*

---

# Step 7: The Complete Notebook Problem

In the original Duckietown notebook, the following scenario is proposed:

* A robot moves in an obstacle-free room. It is equipped with two sensors to measure distances from the walls, allowing it to detect its own position $(x, y)$. The sensors are imperfect, and the manufacturer gives us their inaccuracy ($R$).
* The state is $\mathbf{x} = [x, y]^T$, and the control commands are velocities in each direction, $v_x$ and $v_y$.
* The robot starts at $(0, 0)$ and moves by applying constant commands $v_x = v_y = 1$ for 10 time steps.
* The given matrices are:

$$A = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}, \quad B = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}, \quad Q = \begin{bmatrix} 0.3 & 0 \\ 0 & 0.3 \end{bmatrix}$$


$$H = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}, \quad R = \begin{bmatrix} 0.75 & 0.0 \\ 0.0 & 0.6 \end{bmatrix}$$



We first simulate and visualize the ideal true trajectory, the dynamics-only estimate, and the sensor-only measurements.

### Graph Analysis:

* **Dead Reckoning (blue line):** dynamics alone integrates forward displacement over time, but accumulates error at every step due to wheel noise, drifting away from the true path.
* **Measurements (red dots):** do not accumulate long-term drift, but bounce noticeably around the true line.
* **Our goal with the Kalman Filter:** obtain a better estimate by fusing dynamics and sensor measurements together!

---

# Step 8: Writing and Implementing the `predict` and `update` Functions

In the original notebook code, you find two empty functions with the comment `# TODO Need to update`.
Additionally, the original text recommends:

> *"To multiply a 2D matrix and a 1D vector, use 'np.dot()' and not the '*' operator"*

In modern Python, we can use either:

* The **`@`** operator (e.g., `A @ mu`), which is the native operator for matrix multiplication;
* The **`np.dot(A, mu)`** function.

*(Both perform the proper row-by-column product of linear algebra.)*

---

# Step 9: Running the Kalman Filter on the Robot

Now we run the entire Kalman Filter loop across all time steps of our Duckiebot:

1. Initialize the matrices $\mu_0$, $\Sigma_0$, $A$, $B$, $Q$, $H$, $R$.
2. For each time step:
* Call `predict` to project the position forward using the control command $\mathbf{u}_t = [1, 1]^T$.
* Simulate the sensor reading $\mathbf{z}$.
* Call `update` to fuse the reading and correct the position.
* Store the estimates for the final plot.



---

# Step 10: Analysis and Evaluation of Results

As can be clearly observed from the graph:

1. The trajectory estimated by the **Kalman Filter (green line)** stays very close to the ideal true trajectory (**dashed black line**).
2. It removed the large random fluctuations of the raw measurements (**red crosses**).
3. It does not drift off course like pure odometry (**blue line**).

We can quantitatively measure this improvement by plotting the Euclidean localization error:


$$e_t = \Vert{} \mathbf{x}_{\text{estimated}} - \mathbf{x}_{\text{true}} \Vert{}$$